# ReaVector AI — Kaggle main GPU (T4 ×2)

Personal / owner use only. Pulls one queued job from the Render bridge, runs it on a pinned T4, uploads the GLB back. No public serving.

**GPU 0** = images / LoRA / textures · **GPU 1** = video / 3D.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/rizbiehelp-sys/reavectors.git
%cd reavectors
!pip -q install numpy pillow trimesh requests rembg onnxruntime potracer vtracer
!pip -q install diffusers transformers accelerate safetensors


## Bridge settings
Paste your `AI_BRIDGE_KEY` (from Render → Environment). Keep it in this notebook only — never commit it.

In [ ]:
import os, json, requests
BRIDGE = os.environ.get('BRIDGE_URL', 'https://reavectors.onrender.com')
KEY = os.environ.get('BRIDGE_KEY', 'PASTE_AI_BRIDGE_KEY_HERE')
H = {'X-Bridge-Key': KEY}
def pending():
    r = requests.get(f'{BRIDGE}/api/ai/bridge/jobs', params={'status':'pending'}, headers=H, timeout=30)
    r.raise_for_status(); return r.json()['jobs']
def claim(jid):
    return requests.post(f'{BRIDGE}/api/ai/bridge/jobs/{jid}/claim', headers=H, timeout=30).json()
print('bridge =', BRIDGE, '| key set =', KEY != 'PASTE_AI_BRIDGE_KEY_HERE')
pending()

## Install the 3D model (TripoSR by default)
For SF3D swap the clone/requirements. Hunyuan3D-2.1 (~24 GB) will not fit a T4.

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR /opt/TripoSR
!pip -q install -r /opt/TripoSR/requirements.txt
os.environ['PYTHONPATH'] = '/opt/TripoSR:' + os.environ.get('PYTHONPATH','')

## Run the next queued job
Downloads the job's input image, runs on GPU 1 (3D), uploads the GLB. The website can then load the artifact URL.

In [ ]:
import glob, json, os
_ENV = {'sdxl-base-1.0':'RV_SDXL_PATH','texture-sdxl-inpaint':'RV_TEXTURE_PATH','z-image-turbo':'RV_ZIMAGE_PATH',
        'flux-kontext-dev':'RV_FLUX_PATH','ltx-video':'RV_LTX_PATH','wan2.1-1.3b':'RV_WAN_PATH'}
MERGED = '/kaggle/working/merged'
for _mf in glob.glob('/kaggle/input/**/part.json', recursive=True):
    _src = os.path.dirname(_mf)
    try: _meta = json.load(open(_mf))
    except Exception: continue
    _mid = _meta.get('model_id')
    if _mid not in _ENV: continue
    _dst = os.path.join(MERGED, _mid)
    for _root, _dirs, _files in os.walk(_src):
        _rel = os.path.relpath(_root, _src); _od = os.path.normpath(os.path.join(_dst, _rel))
        os.makedirs(_od, exist_ok=True)
        for _f in _files:
            if _f == 'part.json': continue
            _sp = os.path.join(_root, _f); _dp = os.path.join(_od, _f)
            if not os.path.exists(_dp):
                try: os.symlink(_sp, _dp)
                except Exception:
                    import shutil; shutil.copyfile(_sp, _dp)
    os.environ[_ENV[_mid]] = _dst; print('merged', _mid, '->', _dst, flush=True)

VENV = '/tmp/venv'
NEED3D = any(j.get('task') == 'image2_3d' for j in pending())
import subprocess
def _sh(cmd, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    print('>>', cmd[:90], flush=True)
    print((r.stdout or '')[-400:], (r.stderr or '')[-600:], flush=True)
    return r.returncode
if NEED3D:
    _sh('pip -q install uv')
    _sh('uv venv --python 3.11 --seed /tmp/venv')
    _sh('/tmp/venv/bin/pip install -q --index-url https://download.pytorch.org/whl/cu121 torch==2.5.1')
    _sh('/tmp/venv/bin/pip install -q ninja omegaconf trimesh xatlas huggingface_hub einops pillow numpy potracer vtracer requests')
    _e = os.environ.copy(); _e['TORCH_CUDA_ARCH_LIST'] = '7.5'
_sh('git clone --depth 1 https://github.com/tatsy/torchmcubes.git /tmp/torchmcubes')
    _sh('/tmp/venv/bin/pip install --no-build-isolation /tmp/torchmcubes', env=_e)
    _sh('git clone https://github.com/VAST-AI-Research/TripoSR /opt/TripoSR')
    _sh('/tmp/venv/bin/python -c "import torch, torchmcubes; print(chr(51)+chr(68)+chr(32)+chr(79)+chr(75), torch.__version__)"')
EXT = {'image2_3d':'.glb','text2image':'.png','image_edit':'.png','texture':'.png','text2video':'.mp4','image2video':'.mp4'}
DEADLINE = time.time() + int(os.environ.get('RV_KERNEL_SECONDS', '2700'))
MAX_JOBS = int(os.environ.get('RV_MAX_JOBS', '20'))
done = 0
for _ in range(MAX_JOBS):
    if time.time() > DEADLINE: print('time budget reached'); break
    jobs = pending()
    if not jobs: print('queue empty'); break
    j = claim(jobs[0]['id'])
    if not j: continue
    print('claim', j.get('model'), j.get('task'), flush=True)
    url = j.get('input_url')
    if url and str(url).lower().startswith('http'):
        inp = '/kaggle/working/' + os.path.basename(url.split('?')[0])
        open(inp, 'wb').write(requests.get(url, timeout=120).content)
    else:
        inp = '/kaggle/working/prompt.txt'
        open(inp, 'w').write((j.get('params') or {}).get('prompt', 'a product photo'))
    ext = EXT.get(j.get('task'), '.bin')
    out = '/kaggle/working/result' + ext
    gpu = '1' if j.get('model') in ('triposr','sf3d','hunyuan3d','hunyuan3d-mini','ltx-video','wan2.1') else '0'
    is3d = j.get('task') == 'image2_3d' and os.path.exists(VENV + '/bin/python')
    py = (VENV + '/bin/python') if is3d else 'python'
    env = os.environ.copy()
    if is3d: env['PYTHONPATH'] = '/opt/TripoSR'
    params = json.dumps(j.get('params') or {})
    r = subprocess.run([py, 'scripts/gpu_job.py', '--gpu', gpu, '--model', j['model'], '--image', inp, '--out', out, '--params', params], capture_output=True, text=True, env=env)
    print((r.stdout or '')[-700:], (r.stderr or '')[-700:], flush=True)
    if os.path.exists(out):
        up = requests.post(f"{BRIDGE}/api/ai/bridge/jobs/{j['id']}/result", headers=H, files={'file': open(out, 'rb')}, data={'filename': 'result' + ext}, timeout=300)
        print('upload', up.status_code, flush=True); done += 1; os.remove(out)
    else:
        requests.post(f"{BRIDGE}/api/ai/bridge/jobs/{j['id']}/fail", headers=H, json={'error': (r.stderr or '')[-500:]}, timeout=30)
print('processed', done, 'jobs', flush=True)


## Benchmark each T4 separately
Writes a report (seconds, peak VRAM, failures). Run once per GPU.

In [ ]:
!python scripts/ai_bench.py --gpu 1 --models triposr,sf3d --image /kaggle/working/result.glb 2>/dev/null || true
!python scripts/ai_bench.py --list